In [5]:
# ============================================================================
# Cell 1: 导入库
# ============================================================================
from vnpy.alpha.lab import AlphaLab
from vnpy.trader.constant import Interval
import polars as pl
from pathlib import Path
from datetime import datetime
from vnpy.alpha import Segment, AlphaDataset
import pandas as pd
import numpy as np
import lightgbm as lgb
from factor_define import (
    FACTOR_REGISTRY,
    FACTOR_NAMES
)
import pickle
import gc

In [6]:
# ============================================================================
# Cell 2: 路径配置和AlphaLab创建
# ============================================================================
vt_index_symbol = "000300.SSE"
BASE_PATH = Path('D:/Aquant project/MF')
LAB_PATH = BASE_PATH / 'MF_lab'

# 获取MF_Lab
lab = AlphaLab(str(LAB_PATH))

In [7]:
# ============================================================================
# Cell 3: 时间配置
# ============================================================================
# 总时间跨度
start = datetime(2018, 1, 1)
end = datetime(2026, 3, 31)
interval1 = Interval.MINUTE                  #数据频率

# 回测跨度 回测需要日线数据算收益
test_start = datetime(2025, 1, 1)
test_end = datetime(2026, 3, 31)
interval2 = Interval.DAILY

# 训练跨度
train_start = datetime(2018, 1, 1)
train_end = datetime(2023, 12, 31)

# 验证跨度
valid_start = datetime(2024, 1, 1)
valid_end = datetime(2024, 12, 31)

# 加载成分股代码
component_symbols = lab.load_component_symbols(vt_index_symbol, start, end)

In [8]:
# ============================================================================
# Cell 4: 加载数据集
# ============================================================================
DATASET_NAME = 'v2'
dataset: AlphaDataset = lab.load_dataset(DATASET_NAME)

In [9]:
# ============================================================================
# Cell 4: 从 Dataset 提取 numpy 数据的工具函数
# ============================================================================

def extract_numpy_from_dataset(dataset, segment):
    """
    从 AlphaDataset 提取 numpy 数组给原生 LightGBM 使用

    Parameters
    ----------
    dataset : AlphaDataset
        VNPY 数据集
    segment : Segment
        数据段 (TRAIN, VALID, TEST)

    Returns
    -------
    X : np.ndarray
        特征矩阵
    y : np.ndarray
        标签向量
    df_meta : pl.DataFrame
        包含 datetime 和 vt_symbol 的元数据（用于后续构建信号）
    """
    # 获取学习数据（经过预处理的）
    df = dataset.fetch_learn(segment)

    # 元数据（用于后续生成信号）
    meta_cols = ['datetime', 'vt_symbol']
    df_meta = df.select(meta_cols)

    # 特征列：去掉 datetime, vt_symbol, label
    feature_cols = [c for c in df.columns if c not in ['datetime', 'vt_symbol', 'label']]

    # 转换为 numpy
    X = df.select(feature_cols).to_numpy()
    y = df['label'].to_numpy()

    # 获取日期编码用于分组
    date_codes = df['datetime'].to_numpy()

    # 计算分组大小（每天有多少个样本）
    unique_dates, group_sizes = np.unique(date_codes, return_counts=True)

    print(f'{segment.name}: X.shape={X.shape}, y.shape={y.shape}, de_meta.shape={df_meta.shape}')
    print(f'{segment.name}: 交易日数量 = {len(unique_dates)}, 平均每天样本数 = {group_sizes.mean():.1f}')
    return X, y, df_meta,date_codes, group_sizes

# 提取训练集和验证集数据
print('提取训练数据...')
X_train, y_train, meta_train, date_train, group_train = extract_numpy_from_dataset(dataset, Segment.TRAIN)

print('\n提取验证数据...')
X_valid, y_valid, meta_valid, date_valid, group_valid = extract_numpy_from_dataset(dataset, Segment.VALID)

print('\n提取测试数据...')
X_test, y_test, meta_test, date_test, group_test = extract_numpy_from_dataset(dataset, Segment.TEST)

# 释放 dataset 内存
print('\n释放 dataset 内存...')
del dataset
gc.collect()
print('✅ dataset 已释放')

提取训练数据...
TRAIN: X.shape=(427058, 120), y.shape=(427058,), de_meta.shape=(427058, 2)
TRAIN: 交易日数量 = 1447, 平均每天样本数 = 295.1

提取验证数据...
VALID: X.shape=(71979, 120), y.shape=(71979,), de_meta.shape=(71979, 2)
VALID: 交易日数量 = 242, 平均每天样本数 = 297.4

提取测试数据...
TEST: X.shape=(88410, 120), y.shape=(88410,), de_meta.shape=(88410, 2)
TEST: 交易日数量 = 297, 平均每天样本数 = 297.7

释放 dataset 内存...
✅ dataset 已释放


In [10]:
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings('ignore')
def group_ic_metric(preds, train_data):
    """
    按日期分组计算平均 IC

    参数:
    - preds: 模型预测值
    - train_data: lgb.Dataset 对象，需要预先设置 group 信息

    返回:
    - (metric_name, metric_value, is_higher_better)
    """
    labels = train_data.get_label()

    # 获取分组信息（每天有多少个股票）
    group_sizes = train_data.get_group()

    if group_sizes is None:
        # 如果没有分组信息，计算全局 IC
        ic, _ = spearmanr(preds, labels)
        return 'ic', ic, True

    # 按分组计算 IC
    start_idx = 0
    ics = []

    for size in group_sizes:
        end_idx = start_idx + size

        group_preds = preds[start_idx:end_idx]
        group_labels = labels[start_idx:end_idx]

        # 避免全相同值的情况
        if len(np.unique(group_preds)) > 1 and len(np.unique(group_labels)) > 1:
            try:
                ic, _ = spearmanr(group_preds, group_labels)
                if not np.isnan(ic):
                    ics.append(ic)
            except:
                pass

        start_idx = end_idx

    # 计算平均 IC
    mean_ic = np.mean(ics) if ics else 0
    ic_std = np.std(ics) if ics else 0
    ic_ir = mean_ic / (ic_std + 1e-8)  # IC Information Ratio

    # 可以返回多个指标
    # return [('ic_ir', ic_ir, True),('mean_ic', mean_ic, True)]
    return 'mean_ic', mean_ic, True

In [11]:
# ============================================================================
# Cell 7:
# ============================================================================
print('\n开始训练 LightGBM 模型...')

# 创建 LightGBM 数据集
train_data = lgb.Dataset(X_train, label=y_train)
valid_data = lgb.Dataset(X_valid, label=y_valid, reference=train_data)

# 设置分组信息（关键步骤！）
train_data.set_group(group_train)
valid_data.set_group(group_valid)
print(f'训练集分组数: {len(group_train)}, 总样本: {sum(group_train)}')
print(f'验证集分组数: {len(group_valid)}, 总样本: {sum(group_valid)}')

# 完整的 LightGBM 参数
params = {
    # 目标函数
    'objective': 'regression',
    'metric': '',

    # 提升类型
    'boosting_type': 'gbdt',

    'device': 'gpu',
    # 树结构参数
    'num_leaves': 512,           # 叶子节点数
    'max_depth': -1,            # 树深度（-1表示无限制）
    'min_data_in_leaf': 120,     # 叶节点最小样本数

    # 学习参数
    'learning_rate': 0.005,      # 学习率
    'feature_fraction':0.8879,    # 特征采样比例
    'bagging_fraction': 0.8789,    # 数据采样比例
    'bagging_freq': 5,          # 每5轮迭代进行一次 bagging

    # 正则化参数
    'lambda_l1': 0.1,           # L1 正则化
    'lambda_l2': 1.5,           # L2 正则化

    # 其他参数
    'verbose': -1,
    'seed': 42,
    'num_threads': -1           # 使用所有 CPU 核心
}

# 训练参数
num_boost_round = 10000       # 最大迭代次数
early_stopping_rounds = 100   # 早停轮数

# 训练模型
model = lgb.train(
    params,
    train_data,
    num_boost_round=num_boost_round,
    valid_sets=[train_data, valid_data],
    valid_names=['train', 'valid'],
    feval=group_ic_metric,  # 使用分组 IC 评估
    callbacks=[
        lgb.early_stopping(early_stopping_rounds),
        lgb.log_evaluation(period=1)
    ]
)

print('\n训练完成!')
print(f'最佳迭代轮数: {model.best_iteration}')
print(f'最佳验证 : {model.best_score["valid"]["mean_ic"]:.6f}')


开始训练 LightGBM 模型...
训练集分组数: 1447, 总样本: 427058
验证集分组数: 242, 总样本: 71979
[1]	train's l2: 0.996405	train's mean_ic: 0.105088	valid's l2: 0.996637	valid's mean_ic: 0.0139062
Training until validation scores don't improve for 100 rounds
[2]	train's l2: 0.996198	train's mean_ic: 0.149507	valid's l2: 0.996629	valid's mean_ic: 0.0139185
[3]	train's l2: 0.995996	train's mean_ic: 0.173114	valid's l2: 0.996626	valid's mean_ic: 0.0152379
[4]	train's l2: 0.995791	train's mean_ic: 0.190867	valid's l2: 0.99662	valid's mean_ic: 0.0158102
[5]	train's l2: 0.995586	train's mean_ic: 0.204824	valid's l2: 0.99662	valid's mean_ic: 0.0163573
[6]	train's l2: 0.995383	train's mean_ic: 0.222623	valid's l2: 0.996616	valid's mean_ic: 0.0170539
[7]	train's l2: 0.995179	train's mean_ic: 0.235236	valid's l2: 0.996608	valid's mean_ic: 0.019402
[8]	train's l2: 0.994973	train's mean_ic: 0.245986	valid's l2: 0.996598	valid's mean_ic: 0.0216967
[9]	train's l2: 0.994768	train's mean_ic: 0.252838	valid's l2: 0.996601	valid'

In [12]:
# ============================================================================
# Cell 6: 特征重要性分析
# ============================================================================

print('\n特征重要性分析...')

# 获取特征重要性
importance = model.feature_importance(importance_type='gain')
feature_names = [f'{factor}_lag_{lag}' for factor in FACTOR_NAMES for lag in range(1, 11)]

# 创建重要性 DataFrame
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importance
}).sort_values('importance', ascending=False)

print('Top 20 重要特征:')
print(importance_df.head(120))


特征重要性分析...
Top 20 重要特征:
                         feature    importance
30     corr_close_nextopen_lag_1  31548.771219
31     corr_close_nextopen_lag_2  29722.453187
75            volume_perc5_lag_6  26556.816787
20        corr_ret_lastret_lag_1  26174.739573
70            volume_perc5_lag_1  25777.655098
..                           ...           ...
13           down_vol_perc_lag_4  18006.787822
108  early_corr_volume_ret_lag_9  17981.199487
24        corr_ret_lastret_lag_5  17573.929283
15           down_vol_perc_lag_6  17246.510190
14           down_vol_perc_lag_5  17139.687493

[120 rows x 2 columns]


In [13]:
# ============================================================================
# Cell 7: 生成回测信号
# ============================================================================
print('\n在测试集上预测...')

# 预测
predictions = model.predict(X_test, num_iteration=model.best_iteration)

print(f'预测完成，预测样本数: {len(predictions)}')

# 构建信号 DataFrame
signal = meta_test.with_columns([
    pl.Series('signal', predictions)
])

print(f'\n信号数据形状: {signal.shape}')
print('信号数据预览:')
print(signal.head(10))


在测试集上预测...
预测完成，预测样本数: 88410

信号数据形状: (88410, 3)
信号数据预览:
shape: (10, 3)
┌─────────────────────┬─────────────┬───────────┐
│ datetime            ┆ vt_symbol   ┆ signal    │
│ ---                 ┆ ---         ┆ ---       │
│ datetime[μs]        ┆ str         ┆ f64       │
╞═════════════════════╪═════════════╪═══════════╡
│ 2025-01-02 00:00:00 ┆ 000001.SZSE ┆ -0.01394  │
│ 2025-01-02 00:00:00 ┆ 000002.SZSE ┆ 0.00668   │
│ 2025-01-02 00:00:00 ┆ 000063.SZSE ┆ 0.014234  │
│ 2025-01-02 00:00:00 ┆ 000100.SZSE ┆ 0.004639  │
│ 2025-01-02 00:00:00 ┆ 000157.SZSE ┆ 0.021452  │
│ 2025-01-02 00:00:00 ┆ 000166.SZSE ┆ 0.011839  │
│ 2025-01-02 00:00:00 ┆ 000301.SZSE ┆ -0.004768 │
│ 2025-01-02 00:00:00 ┆ 000333.SZSE ┆ 0.009737  │
│ 2025-01-02 00:00:00 ┆ 000338.SZSE ┆ -0.008384 │
│ 2025-01-02 00:00:00 ┆ 000408.SZSE ┆ -0.033352 │
└─────────────────────┴─────────────┴───────────┘


In [14]:
# ============================================================================
# Cell 8: 保存模型和信号
# ============================================================================
MODEL_NAME = 'v4_z'
SIGNAL_NAME = 'v4_z'

# 保存 LightGBM 模型
MODEL_PICKLE_PATH = LAB_PATH / 'model' / f'{MODEL_NAME}.pkl'
with open(MODEL_PICKLE_PATH, 'wb') as f:
    pickle.dump({
        'model': model,
        'params': params,
        'best_iteration': model.best_iteration,
        'best_score': model.best_score
    }, f)
print(f'模型已保存: {MODEL_PICKLE_PATH}')

# 保存信号
SIGNAL_PARQUET_PATH = LAB_PATH / 'signal' / f'{SIGNAL_NAME}.parquet'
SIGNAL_PARQUET_PATH.parent.mkdir(parents=True, exist_ok=True)
signal.write_parquet(str(SIGNAL_PARQUET_PATH))
print(f'信号已保存: {SIGNAL_PARQUET_PATH}')

模型已保存: D:\Aquant project\MF\MF_lab\model\v4_z.pkl
信号已保存: D:\Aquant project\MF\MF_lab\signal\v4_z.parquet


In [15]:
with pd.option_context('display.max_rows', None):
    print(importance_df.head(120))

                          feature    importance
30      corr_close_nextopen_lag_1  31548.771219
31      corr_close_nextopen_lag_2  29722.453187
75             volume_perc5_lag_6  26556.816787
20         corr_ret_lastret_lag_1  26174.739573
70             volume_perc5_lag_1  25777.655098
33      corr_close_nextopen_lag_4  25510.495599
84             volume_perc6_lag_5  25379.877119
82             volume_perc6_lag_3  25332.721927
72             volume_perc5_lag_3  25092.484026
52             volume_perc3_lag_3  25006.724331
71             volume_perc5_lag_2  24971.413095
34      corr_close_nextopen_lag_5  24943.963880
50             volume_perc3_lag_1  24934.510601
76             volume_perc5_lag_7  24916.231080
32      corr_close_nextopen_lag_3  24615.668294
92             volume_perc7_lag_3  24517.599602
93             volume_perc7_lag_4  24517.265479
38      corr_close_nextopen_lag_9  24331.274613
74             volume_perc5_lag_5  24006.157812
85             volume_perc6_lag_6  23944